# CoherenceBench-IN — GPU Standalone Evaluation (Robust Edition)

**Self-contained. Run on any CUDA GPU PC with VS Code.**

### What this notebook does
1. Installs all required packages
2. Downloads benchmark data from GitHub (~30 MB) with retries
3. Runs up to 5 LLMs with 4-bit quantization (checkpointed — safe to interrupt)
4. Verifies each model's results file is complete (584/584 lines) before moving on
5. Pushes results **directly to GitHub** — no zip/USB transfer needed

### How to use
1. Copy this file to the GPU PC
2. Open in VS Code → select a Python kernel
3. **Run All** — takes ~1–4 hours depending on GPU
4. Results appear in the GitHub repo automatically (Cell 14)

### Requirements
- Python 3.9+ with CUDA GPU (≥ 6 GB VRAM)
- Internet access


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Install packages
# ═══════════════════════════════════════════════════════════
import subprocess, sys

def pip(*args):
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', *args],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'  ⚠️  pip warning: {result.stderr[-300:]}')

print('Installing packages...')

# Detect CUDA version
try:
    nvcc = subprocess.check_output(['nvcc', '--version'], text=True)
    for tag, versions in [('cu124', ['12.4','12.5','12.6']), ('cu121', ['12.1','12.2','12.3']), ('cu118', ['11.8','11.7'])]:
        if any(v in nvcc for v in versions):
            cuda_tag = tag; break
    else:
        cuda_tag = 'cu121'
except Exception:
    cuda_tag = 'cu121'

print(f'  CUDA tag: {cuda_tag}')
pip('torch', '--index-url', f'https://download.pytorch.org/whl/{cuda_tag}')
pip('transformers>=4.45.0', 'accelerate>=0.34.0', 'bitsandbytes>=0.43.0',
    'huggingface_hub>=0.24.0', 'sentencepiece', 'protobuf', 'safetensors')
pip('tiktoken', 'tqdm', 'pandas', 'numpy', 'requests')

# Verify critical imports
import torch, transformers, bitsandbytes
print(f'\n✅ Packages ready.')
print(f'   torch {torch.__version__}  |  transformers {transformers.__version__}  |  bitsandbytes {bitsandbytes.__version__}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Download benchmark data (with retries)
# ═══════════════════════════════════════════════════════════
import requests, json, os, time
from pathlib import Path
from tqdm.auto import tqdm

RAW_BASE    = 'https://raw.githubusercontent.com/jeeth-kataria/coherencebench-in/main'
DATA_DIR    = Path('cb_data/benchmark/english')
RESULTS_DIR = Path('cb_results')
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)


def fetch_with_retry(url: str, retries: int = 5, backoff: float = 2.0) -> bytes:
    """Download URL with exponential backoff retries. Raises on final failure."""
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            return r.content
        except Exception as e:
            if attempt == retries - 1:
                raise RuntimeError(f'Failed after {retries} attempts: {url}\n{e}')
            wait = backoff ** attempt
            print(f'  Retry {attempt+1}/{retries-1} in {wait:.0f}s — {e}')
            time.sleep(wait)


index_path = DATA_DIR / 'index.jsonl'

# Download index
if not index_path.exists():
    print('Downloading benchmark index...')
    index_path.write_bytes(fetch_with_retry(f'{RAW_BASE}/data/benchmark/english/index.jsonl'))

index_meta = [json.loads(l) for l in open(index_path)]
print(f'Index: {len(index_meta)} instances')

# Download instance JSON files (skip existing)
missing = [inst for inst in index_meta if not (DATA_DIR / f"{inst['id']}.json").exists()]
if missing:
    print(f'Downloading {len(missing)} instance files...')
    for inst in tqdm(missing, desc='Downloading', unit='file'):
        dest = DATA_DIR / f"{inst['id']}.json"
        dest.write_bytes(fetch_with_retry(f"{RAW_BASE}/data/benchmark/english/{inst['id']}.json"))
else:
    print('All instance files already present.')

# Verify
actual = sum(1 for _ in DATA_DIR.glob('*.json'))
assert actual == len(index_meta), f'Expected {len(index_meta)} files, found {actual}'
print(f'\n✅ Benchmark ready — {actual} instance files verified.')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Configuration & GPU check
# ═══════════════════════════════════════════════════════════
import torch, re, time
import pandas as pd
import numpy as np

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Check with: nvidia-smi')

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU : {GPU_NAME}')
print(f'VRAM: {GPU_VRAM:.1f} GB')
print(f'PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}')

MAX_CTX_TOKENS = 16_000
MAX_NEW_TOKENS = 15
N_INSTANCES    = 584   # expected total — used for verification

SYSTEM_PROMPT = (
    'You are a precise document analyst. '
    'Read the document carefully and answer the question about its internal consistency. '
    'Your answer must be exactly ONE word: either CONSISTENT or INCONSISTENT. '
    'Do not explain. Do not add punctuation. Output only the single word.'
)
print('\n✅ Configuration ready.')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Load benchmark + helper functions
# ═══════════════════════════════════════════════════════════
import tiktoken, json
enc = tiktoken.get_encoding('cl100k_base')

instances = [json.loads(l) for l in open(DATA_DIR / 'index.jsonl')]
df_bench  = pd.DataFrame(instances)
assert len(instances) == N_INSTANCES, f'Expected {N_INSTANCES} instances, got {len(instances)}'
print(f'Loaded {len(instances)} instances')
print(df_bench.groupby(['dimension', 'answer']).size().unstack(fill_value=0))


def load_text(inst_id: str) -> str:
    with open(DATA_DIR / f'{inst_id}.json') as f:
        return json.load(f)['text']


def build_prompt(text: str, question: str) -> str:
    q_tok  = len(enc.encode(question))
    budget = MAX_CTX_TOKENS - q_tok - 512
    d_tok  = enc.encode(text)
    if len(d_tok) > budget:
        text = enc.decode(d_tok[:budget]) + ' [...document truncated...]'
    return f'Document:\n{text}\n\n---\n{question}'


def parse_answer(response: str) -> str:
    if re.search(r'\bINCONSISTENT\b', response, re.IGNORECASE): return 'INCONSISTENT'
    if re.search(r'\bCONSISTENT\b',   response, re.IGNORECASE): return 'CONSISTENT'
    return 'UNPARSEABLE'


def load_done_ids(results_file: Path) -> set:
    """Read completed IDs from JSONL, skipping any malformed lines."""
    if not results_file.exists():
        return set()
    ids = set()
    with open(results_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                ids.add(json.loads(line)['id'])
            except Exception:
                pass   # skip corrupt lines
    return ids


def verify_results(results_file: Path, expected: int = N_INSTANCES) -> bool:
    """Return True only if the file has exactly `expected` valid lines."""
    ids = load_done_ids(results_file)
    ok  = len(ids) == expected
    status = '✅' if ok else f'⚠️  only {len(ids)}/{expected}'
    print(f'  {results_file.name}: {status}')
    return ok


# Sanity checks
assert parse_answer('INCONSISTENT')              == 'INCONSISTENT'
assert parse_answer('The answer is CONSISTENT.') == 'CONSISTENT'
assert parse_answer("I don't know")              == 'UNPARSEABLE'
print('\n✅ Helpers ready.')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — GPU inference function (robust checkpointing)
# ═══════════════════════════════════════════════════════════
import gc, os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


def run_model(hf_id: str, run_name: str) -> pd.DataFrame:
    """
    Evaluate all benchmark instances through a 4-bit model on CUDA.

    Robustness features:
    - Resumes from last saved instance on re-run
    - os.fsync() after every write  → data guaranteed on disk
    - Skips OOM/error instances (records as UNPARSEABLE) instead of crashing
    - Verifies final count == 584 and raises if incomplete
    """
    results_file = RESULTS_DIR / f'{run_name}_results.jsonl'
    done_ids     = load_done_ids(results_file)
    remaining    = [i for i in instances if i['id'] not in done_ids]

    if not remaining:
        print(f'  {run_name}: already complete ({len(instances)} instances).')
    else:
        if done_ids:
            print(f'  Resuming {run_name} — {len(done_ids)} done, {len(remaining)} to go.')
        else:
            print(f'  Starting {run_name} — {len(remaining)} instances.')

        print(f'  Loading tokenizer: {hf_id}')
        tokenizer = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id

        quant_cfg = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_compute_dtype    = torch.float16,
            bnb_4bit_quant_type       = 'nf4',
            bnb_4bit_use_double_quant = True,
        )
        print(f'  Loading model in 4-bit on GPU...')
        model = AutoModelForCausalLM.from_pretrained(
            hf_id,
            quantization_config = quant_cfg,
            device_map          = 'auto',
            trust_remote_code   = True,
        )
        model.eval()
        used_gb = torch.cuda.memory_allocated(0) / 1024**3
        print(f'  Model loaded. VRAM used: {used_gb:.1f} GB')
        print(f'  Evaluating {len(remaining)} instances...')

        unparseable = skipped = 0
        interrupted = False

        with open(results_file, 'a', buffering=1) as out_f:   # line-buffered
            fd = out_f.fileno()
            for inst in tqdm(remaining, desc=run_name, unit='inst', dynamic_ncols=True):
                try:
                    text     = load_text(inst['id'])
                    user_msg = build_prompt(text, inst['question'])
                    messages = [
                        {'role': 'system', 'content': SYSTEM_PROMPT},
                        {'role': 'user',   'content': user_msg},
                    ]
                    chat_str = tokenizer.apply_chat_template(
                        messages, tokenize=False, add_generation_prompt=True
                    )
                    inputs = tokenizer(chat_str, return_tensors='pt').to('cuda')

                    # Hard-clamp to context window
                    max_len = MAX_CTX_TOKENS + 512
                    if inputs['input_ids'].shape[1] > max_len:
                        inputs = {k: v[:, -max_len:] for k, v in inputs.items()}

                    with torch.no_grad():
                        out = model.generate(
                            **inputs,
                            max_new_tokens = MAX_NEW_TOKENS,
                            do_sample      = False,
                            pad_token_id   = tokenizer.pad_token_id,
                        )
                    new_tok  = out[0][inputs['input_ids'].shape[1]:]
                    response = tokenizer.decode(new_tok, skip_special_tokens=True).strip()
                    pred     = parse_answer(response)
                    correct  = (pred == inst['answer'])
                    if pred == 'UNPARSEABLE':
                        unparseable += 1

                except KeyboardInterrupt:
                    print(f'\n  Interrupted. {len(load_done_ids(results_file))} instances saved.')
                    interrupted = True
                    break

                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache(); gc.collect()
                    response, pred, correct = 'OOM', 'UNPARSEABLE', False
                    skipped += 1

                except Exception as e:
                    response, pred, correct = str(e)[:80], 'UNPARSEABLE', False
                    skipped += 1

                record = json.dumps({
                    'id':             inst['id'],
                    'dimension':      inst['dimension'],
                    'distance':       inst['distance'],
                    'context_tokens': inst['context_tokens'],
                    'gold':           inst['answer'],
                    'pred':           pred,
                    'correct':        correct,
                    'raw_response':   response[:120],
                })
                out_f.write(record + '\n')
                os.fsync(fd)   # ← flush kernel buffer to disk after every line

        del model
        gc.collect()
        torch.cuda.empty_cache()

        if not interrupted:
            print(f'  ✅ {run_name} done. Unparseable: {unparseable}, OOM/errors: {skipped}')

    # ── Verify completeness ─────────────────────────────────────
    saved = load_done_ids(results_file)
    if len(saved) < N_INSTANCES:
        print(f'  ⚠️  {run_name}: only {len(saved)}/{N_INSTANCES} saved. Re-run this cell to resume.')
    else:
        print(f'  ✅ {run_name}: {len(saved)}/{N_INSTANCES} instances verified on disk.')

    # ── Load and summarise ──────────────────────────────────────
    df = pd.DataFrame([json.loads(l) for l in open(results_file)])
    acc = df['correct'].mean()
    print(f'\n  {run_name}  overall accuracy: {acc:.1%}  ({df["correct"].sum()}/{len(df)})')
    for dim, sub in df.groupby('dimension'):
        print(f'    {dim:<28}: {sub["correct"].mean():.1%}')
    return df


print('✅ run_model() ready.')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Model 1: Llama-3.2-3B  (~2.1 GB VRAM, ~20–40 min)
# ═══════════════════════════════════════════════════════════
df_llama3 = run_model(
    hf_id    = 'unsloth/Llama-3.2-3B-Instruct',
    run_name = 'llama3_3b',
)


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Model 2: Qwen2.5-7B  (~5.2 GB VRAM, ~40–80 min)
# ═══════════════════════════════════════════════════════════
df_qwen7 = run_model(
    hf_id    = 'Qwen/Qwen2.5-7B-Instruct',
    run_name = 'qwen25_7b',
)


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — Model 3: Mistral-7B  (~5.5 GB VRAM, ~40–80 min)
# ═══════════════════════════════════════════════════════════
df_mistral = run_model(
    hf_id    = 'mistralai/Mistral-7B-Instruct-v0.3',
    run_name = 'mistral_7b',
)


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — Model 4: Phi-3.5-mini  (~2.5 GB VRAM, ~25–45 min)
# ═══════════════════════════════════════════════════════════
df_phi = run_model(
    hf_id    = 'microsoft/Phi-3.5-mini-instruct',
    run_name = 'phi35_mini',
)


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — Model 5: Qwen2.5-14B  (~8.5 GB VRAM, ~60–120 min)
# ═══════════════════════════════════════════════════════════
df_qwen14 = run_model(
    hf_id    = 'Qwen/Qwen2.5-14B-Instruct',
    run_name = 'qwen25_14b',
)


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — Verify all results & print summary table
# ═══════════════════════════════════════════════════════════
MODELS = {
    'Llama-3.2-3B':  'llama3_3b',
    'Qwen2.5-7B':    'qwen25_7b',
    'Mistral-7B':    'mistral_7b',
    'Phi-3.5-mini':  'phi35_mini',
    'Qwen2.5-14B':   'qwen25_14b',
}
DIMS      = ['entity_consistency', 'temporal_coherence', 'causal_chain']
DIM_SHORT = {'entity_consistency': 'Entity', 'temporal_coherence': 'Temporal', 'causal_chain': 'Causal'}

print('Results verification:')
all_ok = True
rows   = []
for display, fname in MODELS.items():
    fpath = RESULTS_DIR / f'{fname}_results.jsonl'
    if not fpath.exists():
        print(f'  {display:<18}: ⬜ not run')
        continue
    ids = load_done_ids(fpath)
    status = '✅' if len(ids) == N_INSTANCES else f'⚠️  {len(ids)}/{N_INSTANCES}'
    print(f'  {display:<18}: {status}')
    if len(ids) < N_INSTANCES:
        all_ok = False
        continue
    df = pd.DataFrame([json.loads(l) for l in open(fpath)])
    row = {'Model': display}
    for dim in DIMS:
        sub = df[df['dimension'] == dim]
        row[DIM_SHORT[dim]] = f"{sub['correct'].mean():.1%}" if len(sub) else '—'
    row['Overall'] = f"{df['correct'].mean():.1%}"
    rows.append(row)

if rows:
    summary = pd.DataFrame(rows).set_index('Model')
    summary.to_csv(RESULTS_DIR / 'table3_accuracy_by_dimension.csv')
    print('\nTable 3 — Accuracy by Dimension')
    print('=' * 55)
    print(summary.to_string())
    print(f'\nSaved: {RESULTS_DIR}/table3_accuracy_by_dimension.csv')
else:
    print('\nNo complete model results yet.')

if not all_ok:
    print('\n⚠️  Some models are incomplete — re-run their cells before pushing.')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 12 — Zip results (with integrity check)
# ═══════════════════════════════════════════════════════════
import zipfile
from pathlib import Path

zip_path = Path('cb_results.zip')
files_added = []

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(RESULTS_DIR.iterdir()):
        if f.suffix in ('.jsonl', '.csv'):
            zf.write(f, f.name)
            files_added.append(f.name)
            print(f'  + {f.name}  ({f.stat().st_size / 1024:.0f} KB)')

# Verify the zip is readable
bad = zipfile.ZipFile(zip_path).testzip()
if bad:
    raise RuntimeError(f'Corrupt file in zip: {bad}')

size_mb = zip_path.stat().st_size / 1024 / 1024
print(f'\n✅ {zip_path.resolve()}  ({size_mb:.2f} MB) — integrity OK')
print(f'   {len(files_added)} files: {files_added}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 13 — Push results directly to GitHub (no USB needed)
# ═══════════════════════════════════════════════════════════
# Instructions:
#   1. Go to https://github.com/settings/tokens/new
#   2. Name: "coherencebench-results"  |  Expiration: 7 days
#   3. Scopes: check "repo" (or just "public_repo" if repo is public)
#   4. Click Generate token → paste below
#
# Only JSONL files are pushed (no large model files).
# Run this cell once after all models complete.

import requests, base64, json

GITHUB_TOKEN = ''   # ← paste your PAT here: ghp_...
REPO         = 'jeeth-kataria/coherencebench-in'
BRANCH       = 'main'

if not GITHUB_TOKEN:
    print('⚠️  Set GITHUB_TOKEN above and re-run this cell.')
else:
    headers = {
        'Authorization': f'token {GITHUB_TOKEN}',
        'Accept':        'application/vnd.github.v3+json',
    }
    api = f'https://api.github.com/repos/{REPO}/contents'

    for f in sorted(RESULTS_DIR.glob('*.jsonl')):
        remote_path = f'results/{f.name}'
        content_b64 = base64.b64encode(f.read_bytes()).decode()

        # Get existing SHA (needed for updates)
        r = requests.get(f'{api}/{remote_path}', headers=headers)
        sha = r.json().get('sha') if r.ok else None

        payload = {
            'message': f'Add GPU results: {f.name}',
            'content': content_b64,
            'branch':  BRANCH,
        }
        if sha:
            payload['sha'] = sha

        r = requests.put(f'{api}/{remote_path}', headers=headers, json=payload)
        status = '✅ pushed' if r.ok else f'❌ {r.status_code} {r.json().get("message","")}'
        print(f'  {f.name}: {status}')

    # Also push the CSV summary
    for f in sorted(RESULTS_DIR.glob('*.csv')):
        remote_path = f'results/{f.name}'
        content_b64 = base64.b64encode(f.read_bytes()).decode()
        r_get = requests.get(f'{api}/{remote_path}', headers=headers)
        sha = r_get.json().get('sha') if r_get.ok else None
        payload = {'message': f'Add {f.name}', 'content': content_b64, 'branch': BRANCH}
        if sha: payload['sha'] = sha
        r = requests.put(f'{api}/{remote_path}', headers=headers, json=payload)
        print(f'  {f.name}: {"✅ pushed" if r.ok else "❌ " + str(r.status_code)}')

    print('\nDone! Results are now live at:')
    print(f'  https://github.com/{REPO}/tree/{BRANCH}/results')
